In [1]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "8"

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    classification_report
)

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv(
    "Data/data_processed.csv"
)

X = df.drop(columns=["Class"])
y = df["Class"]

print(y.value_counts())

Class
0    283253
1       473
Name: count, dtype: int64


In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test :", X_test.shape)

Train: (226980, 30)
Valid: (28373, 30)
Test : (28373, 30)


In [4]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_valid_scaled = scaler.transform(X_valid)

X_test_scaled = scaler.transform(X_test)

In [5]:
smote = SMOTE(
    sampling_strategy=0.3,
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

print(y_train_smote.value_counts())

Class
0    226602
1     67980
Name: count, dtype: int64


In [6]:
class_ratio = (
    y_train_smote.value_counts()[0]
    /
    y_train_smote.value_counts()[1]
)

print(class_ratio)

3.3333627537511035


In [7]:
model = XGBClassifier(
    n_estimators=1000,

    learning_rate=0.20,

    max_depth=5,
    min_child_weight=5,

    subsample=0.779,
    colsample_bytree=0.893,

    gamma=1.48,

    reg_alpha=1.29,
    reg_lambda=1.18,

    eval_metric="aucpr",

    random_state=42,
    n_jobs=-1,

    scale_pos_weight=class_ratio,

    early_stopping_rounds=50
)

model.fit(
    X_train_smote,
    y_train_smote,

    eval_set=[
        (
            X_valid_scaled,
            y_valid
        )
    ],

    verbose=True
)

[0]	validation_0-aucpr:0.47408
[1]	validation_0-aucpr:0.49503
[2]	validation_0-aucpr:0.63051
[3]	validation_0-aucpr:0.64815
[4]	validation_0-aucpr:0.66319
[5]	validation_0-aucpr:0.66219
[6]	validation_0-aucpr:0.66012
[7]	validation_0-aucpr:0.66854
[8]	validation_0-aucpr:0.67521
[9]	validation_0-aucpr:0.67898
[10]	validation_0-aucpr:0.67998
[11]	validation_0-aucpr:0.68579
[12]	validation_0-aucpr:0.68667
[13]	validation_0-aucpr:0.68902
[14]	validation_0-aucpr:0.68968
[15]	validation_0-aucpr:0.69096
[16]	validation_0-aucpr:0.69545
[17]	validation_0-aucpr:0.69775
[18]	validation_0-aucpr:0.68978
[19]	validation_0-aucpr:0.68611
[20]	validation_0-aucpr:0.68924
[21]	validation_0-aucpr:0.68710
[22]	validation_0-aucpr:0.68912
[23]	validation_0-aucpr:0.69166
[24]	validation_0-aucpr:0.69273
[25]	validation_0-aucpr:0.73403
[26]	validation_0-aucpr:0.74886
[27]	validation_0-aucpr:0.74577
[28]	validation_0-aucpr:0.74471
[29]	validation_0-aucpr:0.71213
[30]	validation_0-aucpr:0.71281
[31]	validation_0-

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.893, device=None, early_stopping_rounds=50,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=1.48, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=-1, num_parallel_tree=None, ...)

In [8]:
y_pred = model.predict(
    X_test_scaled
)

y_prob = model.predict_proba(
    X_test_scaled
)[:, 1]

In [9]:
print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)

              precision    recall  f1-score   support

           0     0.9998    0.9991    0.9994     28326
           1     0.6154    0.8511    0.7143        47

    accuracy                         0.9989     28373
   macro avg     0.8076    0.9251    0.8569     28373
weighted avg     0.9991    0.9989    0.9990     28373

